In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
plt.rcParams.update({
    "figure.facecolor":  "#FAFAFA",
    "axes.facecolor":    "#FFFFFF",
    "axes.grid":         True,
    "grid.color":        "#E8E8E8",
    "grid.linewidth":    0.6,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.spines.left":  True,
    "axes.spines.bottom":True,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "font.family":       "DejaVu Sans",
    "figure.dpi":        130,
})
 
BLUE    = "#2563EB"
ORANGE  = "#F59E0B"
GREEN   = "#10B981"
RED     = "#EF4444"
PURPLE  = "#8B5CF6"
TEAL    = "#06B6D4"
PINK    = "#EC4899"
GRAY    = "#6B7280"
PALETTE = [BLUE, ORANGE, GREEN, RED, PURPLE, TEAL, PINK, GRAY]
 
OUT = "eda_outputs/plots"
import os; os.makedirs(OUT, exist_ok=True)

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin(np.radians(lat2 - lat1) / 2) ** 2
         + np.cos(phi1) * np.cos(phi2)
         * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
 

In [5]:
print("Loading data...")
df = pd.read_csv("../data/raw/uber_fares.csv")
print(f"  Raw shape: {df.shape}")
 
# Clean
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], utc=True, errors="coerce")
df["fare_amount"]     = pd.to_numeric(df["fare_amount"],     errors="coerce")
df["passenger_count"] = pd.to_numeric(df["passenger_count"], errors="coerce")
for c in ["pickup_latitude","pickup_longitude","dropoff_latitude","dropoff_longitude"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
 
df = df.dropna(subset=["fare_amount","pickup_datetime",
                        "pickup_latitude","pickup_longitude",
                        "dropoff_latitude","dropoff_longitude"])
fare_upper = df["fare_amount"].quantile(0.999)
df = df[(df["fare_amount"] >= 2.50) & (df["fare_amount"] <= fare_upper)]
df = df[
    df["pickup_latitude"].between(40.4774, 41.0176) &
    df["pickup_longitude"].between(-74.2591, -73.6004) &
    df["dropoff_latitude"].between(40.4774, 41.0176) &
    df["dropoff_longitude"].between(-74.2591, -73.6004)
]
df["passenger_count"] = df["passenger_count"].fillna(1).clip(1, 6).astype(int)
df = df.reset_index(drop=True)
 
# Feature extraction
df["hour"]       = df["pickup_datetime"].dt.hour
df["dow"]        = df["pickup_datetime"].dt.dayofweek
df["month"]      = df["pickup_datetime"].dt.month
df["year"]       = df["pickup_datetime"].dt.year
df["distance_km"]= haversine(
    df["pickup_latitude"].values,  df["pickup_longitude"].values,
    df["dropoff_latitude"].values, df["dropoff_longitude"].values
)
df["is_rush"]    = ((df["hour"].between(7,9)) | (df["hour"].between(17,19))).astype(int)
df["is_night"]   = ((df["hour"] >= 22) | (df["hour"] <= 4)).astype(int)
df["is_weekend"] = (df["dow"] >= 5).astype(int)
df["log_fare"]   = np.log1p(df["fare_amount"])
 
DOW_LABELS   = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
 
print(f"  Clean shape: {df.shape}")
print(f"  Fare mean: ${df['fare_amount'].mean():.2f}  std: ${df['fare_amount'].std():.2f}")
print(f"  Distance mean: {df['distance_km'].mean():.2f} km\n")
 

Loading data...
  Raw shape: (55000, 8)
  Clean shape: (54945, 17)
  Fare mean: $43.26  std: $22.87
  Distance mean: 16.84 km



In [6]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 1 — Fare Distribution (raw vs log)
# ══════════════════════════════════════════════════════════════════════
print("Plot 1: Fare Distribution...")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Fare Amount Distribution", fontsize=15, fontweight="bold", y=1.01)
 
# Raw
axes[0].hist(df["fare_amount"], bins=100, color=BLUE, edgecolor="white",
             linewidth=0.3, alpha=0.9)
axes[0].axvline(df["fare_amount"].mean(),   color=RED,    lw=2, ls="--", label=f"Mean  ${df['fare_amount'].mean():.1f}")
axes[0].axvline(df["fare_amount"].median(), color=ORANGE, lw=2, ls=":",  label=f"Median ${df['fare_amount'].median():.1f}")
axes[0].set_title("Raw Fare Amount")
axes[0].set_xlabel("Fare ($)")
axes[0].set_ylabel("Count")
axes[0].legend(framealpha=0.9)
 
# Stats box
stats_text = (f"Mean:    ${df['fare_amount'].mean():.2f}\n"
              f"Median: ${df['fare_amount'].median():.2f}\n"
              f"Std:       ${df['fare_amount'].std():.2f}\n"
              f"Min:       ${df['fare_amount'].min():.2f}\n"
              f"Max:      ${df['fare_amount'].max():.2f}")
axes[0].text(0.97, 0.97, stats_text, transform=axes[0].transAxes,
             fontsize=8.5, va="top", ha="right",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="#EFF6FF",
                       edgecolor=BLUE, alpha=0.9))
 
# Log-transformed
axes[1].hist(df["log_fare"], bins=100, color=GREEN, edgecolor="white",
             linewidth=0.3, alpha=0.9)
axes[1].axvline(df["log_fare"].mean(),   color=RED,    lw=2, ls="--", label=f"Mean  {df['log_fare'].mean():.2f}")
axes[1].axvline(df["log_fare"].median(), color=ORANGE, lw=2, ls=":",  label=f"Median {df['log_fare'].median():.2f}")
axes[1].set_title("Log(1 + Fare) — Normalised")
axes[1].set_xlabel("log1p(Fare)")
axes[1].set_ylabel("Count")
axes[1].legend(framealpha=0.9)
axes[1].text(0.03, 0.97,
             "Right skew removed\nafter log transform",
             transform=axes[1].transAxes, fontsize=8.5, va="top",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#F0FDF4",
                       edgecolor=GREEN, alpha=0.9))
 
fig.tight_layout()
fig.savefig(f"{OUT}/01_fare_distribution.png", bbox_inches="tight")
plt.close(fig)
 

Plot 1: Fare Distribution...


In [7]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 2 — Fare by Hour of Day
# ══════════════════════════════════════════════════════════════════════
print("Plot 2: Fare by Hour...")
hourly = df.groupby("hour")["fare_amount"].agg(["mean","median","std","count"]).reset_index()
 
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hourly["hour"], hourly["mean"], color=BLUE, alpha=0.75,
              width=0.6, label="Mean fare", zorder=2)
ax.plot(hourly["hour"], hourly["median"], "o-", color=ORANGE, lw=2.5,
        ms=5, label="Median fare", zorder=3)
 
# Shade rush hours
ax.axvspan(7,  9,  alpha=0.12, color=RED,    label="AM Rush (7–9)")
ax.axvspan(17, 19, alpha=0.12, color=PURPLE, label="PM Rush (17–19)")
ax.axvspan(22, 24, alpha=0.08, color=TEAL,   label="Late Night")
ax.axvspan(0,   4, alpha=0.08, color=TEAL)
 
# Annotate peak bar
peak_h = hourly.loc[hourly["mean"].idxmax()]
ax.annotate(f"Peak\n${peak_h['mean']:.1f}",
            xy=(peak_h["hour"], peak_h["mean"]),
            xytext=(peak_h["hour"] + 1.5, peak_h["mean"] + 2),
            fontsize=8.5, color=RED, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.2))
 
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right")
ax.set_title("Average Fare by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Fare ($)")
ax.legend(loc="upper left", framealpha=0.9, fontsize=9)
 
fig.tight_layout()
fig.savefig(f"{OUT}/02_fare_by_hour.png", bbox_inches="tight")
plt.close(fig)
 
 

Plot 2: Fare by Hour...


In [8]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 3 — Fare by Day of Week
# ══════════════════════════════════════════════════════════════════════
print("Plot 3: Fare by Day of Week...")
dow_stats = df.groupby("dow")["fare_amount"].agg(["mean","median","count"]).reset_index()
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Fare Patterns by Day of Week", fontsize=14, fontweight="bold")
 
colors_dow = [RED if d >= 5 else BLUE for d in dow_stats["dow"]]
axes[0].bar([DOW_LABELS[d] for d in dow_stats["dow"]], dow_stats["mean"],
            color=colors_dow, edgecolor="white", linewidth=0.5, alpha=0.9)
axes[0].set_title("Mean Fare  (red = weekend)")
axes[0].set_ylabel("Mean Fare ($)")
axes[0].set_xlabel("Day")
for i, v in enumerate(dow_stats["mean"]):
    axes[0].text(i, v + 0.2, f"${v:.1f}", ha="center", fontsize=8, fontweight="bold")
 
# Trip volume
axes[1].bar([DOW_LABELS[d] for d in dow_stats["dow"]], dow_stats["count"],
            color=colors_dow, edgecolor="white", linewidth=0.5, alpha=0.9)
axes[1].set_title("Trip Volume by Day  (red = weekend)")
axes[1].set_ylabel("Number of Trips")
axes[1].set_xlabel("Day")
 
fig.tight_layout()
fig.savefig(f"{OUT}/03_fare_by_dow.png", bbox_inches="tight")
plt.close(fig)

Plot 3: Fare by Day of Week...


In [9]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 4 — Fare by Month
# ══════════════════════════════════════════════════════════════════════
print("Plot 4: Fare by Month...")
monthly = df.groupby("month")["fare_amount"].agg(["mean","median","count"]).reset_index()
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Fare & Volume by Month", fontsize=14, fontweight="bold")
 
axes[0].plot(monthly["month"], monthly["mean"], "o-", color=BLUE, lw=2.5, ms=8, label="Mean")
axes[0].plot(monthly["month"], monthly["median"], "s--", color=ORANGE, lw=2, ms=7, label="Median")
axes[0].fill_between(monthly["month"], monthly["mean"], alpha=0.12, color=BLUE)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(MONTH_LABELS)
axes[0].set_title("Mean vs Median Fare by Month")
axes[0].set_ylabel("Fare ($)")
axes[0].legend()
 
# Heatmap by month × hour (mean fare)
pivot = df.pivot_table(values="fare_amount", index="hour", columns="month", aggfunc="mean")
im = axes[1].imshow(pivot, aspect="auto", cmap="YlOrRd", interpolation="nearest")
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(MONTH_LABELS, rotation=45)
axes[1].set_yticks(range(0, 24, 3))
axes[1].set_yticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
axes[1].set_title("Mean Fare Heatmap  (Hour × Month)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Hour of Day")
plt.colorbar(im, ax=axes[1], label="Mean Fare ($)", shrink=0.85)
 
fig.tight_layout()
fig.savefig(f"{OUT}/04_fare_by_month.png", bbox_inches="tight")
plt.close(fig)
 

Plot 4: Fare by Month...


In [10]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 5 — Distance vs Fare Scatter
# ══════════════════════════════════════════════════════════════════════
print("Plot 5: Distance vs Fare...")
sample = df.sample(min(8000, len(df)), random_state=42)
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Distance vs Fare Relationship", fontsize=14, fontweight="bold")
 
# Coloured by hour
sc = axes[0].scatter(sample["distance_km"], sample["fare_amount"],
                     c=sample["hour"], cmap="plasma",
                     alpha=0.35, s=6)
plt.colorbar(sc, ax=axes[0], label="Hour of Day")
# Trend line
z = np.polyfit(sample["distance_km"], sample["fare_amount"], 1)
x_line = np.linspace(0, sample["distance_km"].quantile(0.99), 100)
axes[0].plot(x_line, np.polyval(z, x_line), color=RED, lw=2.5,
             label=f"Trend: ${z[0]:.2f}/km + ${z[1]:.2f}")
axes[0].set_xlim(0, sample["distance_km"].quantile(0.99))
axes[0].set_ylim(0, sample["fare_amount"].quantile(0.99))
axes[0].set_title("Coloured by Hour of Day")
axes[0].set_xlabel("Distance (km)")
axes[0].set_ylabel("Fare ($)")
axes[0].legend(fontsize=9)
corr = df["distance_km"].corr(df["fare_amount"])
axes[0].text(0.03, 0.97, f"Pearson r = {corr:.3f}", transform=axes[0].transAxes,
             fontsize=9, va="top",
             bbox=dict(boxstyle="round", facecolor="#EFF6FF", edgecolor=BLUE))
 
# Coloured by rush hour
colors_rush = [RED if r else BLUE for r in sample["is_rush"]]
axes[1].scatter(sample["distance_km"], sample["fare_amount"],
                c=colors_rush, alpha=0.3, s=6)
from matplotlib.lines import Line2D
legend_els = [Line2D([0],[0],marker="o",color="w",markerfacecolor=RED,  ms=7, label="Rush Hour"),
              Line2D([0],[0],marker="o",color="w",markerfacecolor=BLUE, ms=7, label="Off-Peak")]
axes[1].legend(handles=legend_els, fontsize=9)
axes[1].set_xlim(0, sample["distance_km"].quantile(0.99))
axes[1].set_ylim(0, sample["fare_amount"].quantile(0.99))
axes[1].set_title("Rush Hour vs Off-Peak Pricing")
axes[1].set_xlabel("Distance (km)")
axes[1].set_ylabel("Fare ($)")
 
fig.tight_layout()
fig.savefig(f"{OUT}/05_distance_vs_fare.png", bbox_inches="tight")
plt.close(fig)
 

Plot 5: Distance vs Fare...


In [11]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 6 — Pickup Density Heatmap
# ══════════════════════════════════════════════════════════════════════
print("Plot 6: Pickup Heatmap...")
sample_map = df.sample(min(20000, len(df)), random_state=7)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("NYC Ride Geographic Density", fontsize=14, fontweight="bold")
 
for ax, lat_col, lon_col, title in [
    (axes[0], "pickup_latitude",  "pickup_longitude",  "Pickup  Density"),
    (axes[1], "dropoff_latitude", "dropoff_longitude", "Dropoff Density"),
]:
    h = ax.hist2d(sample_map[lon_col], sample_map[lat_col],
                  bins=100, cmap="hot_r",
                  range=[[-74.10, -73.72], [40.57, 40.92]])
    plt.colorbar(h[3], ax=ax, label="Trip Count", shrink=0.85)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
 
    # Annotate airports
    for name, lat, lon in [("JFK", 40.641, -73.778),
                            ("LGA", 40.777, -73.874),
                            ("EWR", 40.690, -74.174)]:
        if ax.get_xlim()[0] < lon < ax.get_xlim()[1]:
            ax.annotate(name, xy=(lon, lat), fontsize=7.5, color="white",
                        fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2",
                                  facecolor="black", alpha=0.5))
 
fig.tight_layout()
fig.savefig(f"{OUT}/06_geo_density.png", bbox_inches="tight")
plt.close(fig)
 

Plot 6: Pickup Heatmap...


In [12]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 7 — Passenger Count Analysis
# ══════════════════════════════════════════════════════════════════════
print("Plot 7: Passenger Count...")
pax_counts = df["passenger_count"].value_counts().sort_index()
pax_fare   = df.groupby("passenger_count")["fare_amount"].agg(["mean","median"])
 
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle("Passenger Count Analysis", fontsize=14, fontweight="bold")
 
# Pie
axes[0].pie(pax_counts.values,
            labels=[f"{p} pax\n({v/pax_counts.sum()*100:.1f}%)" for p, v in pax_counts.items()],
            colors=PALETTE[:len(pax_counts)], startangle=140,
            wedgeprops=dict(edgecolor="white", linewidth=1.5),
            textprops=dict(fontsize=8.5))
axes[0].set_title("Trip Share by Passengers")
 
# Bar — count
axes[1].bar(pax_counts.index, pax_counts.values, color=BLUE,
            edgecolor="white", linewidth=0.5, alpha=0.9)
axes[1].set_title("Trip Volume by Passengers")
axes[1].set_xlabel("Passenger Count")
axes[1].set_ylabel("Trips")
axes[1].set_xticks(range(1, 7))
 
# Bar — mean fare
axes[2].bar(pax_fare.index, pax_fare["mean"], color=ORANGE,
            edgecolor="white", linewidth=0.5, alpha=0.9, label="Mean")
axes[2].plot(pax_fare.index, pax_fare["median"], "o-", color=RED,
             ms=6, lw=2, label="Median")
axes[2].set_title("Mean Fare by Passenger Count")
axes[2].set_xlabel("Passenger Count")
axes[2].set_ylabel("Fare ($)")
axes[2].set_xticks(range(1, 7))
axes[2].legend()
 
fig.tight_layout()
fig.savefig(f"{OUT}/07_passenger_analysis.png", bbox_inches="tight")
plt.close(fig)
 

Plot 7: Passenger Count...


In [13]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 8 — Fare by Year (trend)
# ══════════════════════════════════════════════════════════════════════
print("Plot 8: Yearly Trend...")
yearly = df.groupby("year")["fare_amount"].agg(["mean","median","count","std"]).reset_index()
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Fare Trends Over Time (2009–2015)", fontsize=14, fontweight="bold")
 
axes[0].plot(yearly["year"], yearly["mean"],   "o-", color=BLUE,   lw=2.5, ms=8, label="Mean")
axes[0].plot(yearly["year"], yearly["median"], "s--", color=ORANGE, lw=2,   ms=7, label="Median")
axes[0].fill_between(yearly["year"],
                     yearly["mean"] - yearly["std"],
                     yearly["mean"] + yearly["std"],
                     alpha=0.12, color=BLUE, label="±1 std")
axes[0].set_title("Mean Fare by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Fare ($)")
axes[0].set_xticks(yearly["year"])
axes[0].legend()
 
axes[1].bar(yearly["year"], yearly["count"], color=GREEN,
            edgecolor="white", linewidth=0.5, alpha=0.9)
axes[1].set_title("Trip Volume by Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Trips")
axes[1].set_xticks(yearly["year"])
for i, (y, c) in enumerate(zip(yearly["year"], yearly["count"])):
    axes[1].text(y, c + 20, str(c), ha="center", fontsize=8)
 
fig.tight_layout()
fig.savefig(f"{OUT}/08_yearly_trend.png", bbox_inches="tight")
plt.close(fig)
 

Plot 8: Yearly Trend...


In [14]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 9 — Rush Hour vs Off-Peak Boxplots
# ══════════════════════════════════════════════════════════════════════
print("Plot 9: Rush vs Off-Peak...")
df["period"] = "Off-Peak"
df.loc[df["is_rush"]==1,  "period"] = "Rush Hour"
df.loc[df["is_night"]==1, "period"] = "Late Night"
df.loc[df["is_weekend"]==1, "period"] = "Weekend"
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fare Distribution by Time Period", fontsize=14, fontweight="bold")
 
order = ["Off-Peak","Rush Hour","Late Night","Weekend"]
colors_bp = [BLUE, RED, TEAL, ORANGE]
 
# Boxplot
bp = axes[0].boxplot(
    [df.loc[df["period"]==p, "fare_amount"].values for p in order],
    labels=order, patch_artist=True, notch=True,
    medianprops=dict(color="white", linewidth=2.5),
)
for patch, col in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[0].set_title("Fare Boxplot by Period")
axes[0].set_ylabel("Fare ($)")
axes[0].tick_params(axis="x", rotation=10)
 
# Violin
parts = axes[1].violinplot(
    [df.loc[df["period"]==p, "fare_amount"].sample(min(3000,len(df[df["period"]==p])),
                                                    random_state=1).values for p in order],
    positions=range(len(order)), showmedians=True, showextrema=False,
)
for i, (body, col) in enumerate(zip(parts["bodies"], colors_bp)):
    body.set_facecolor(col); body.set_alpha(0.65)
parts["cmedians"].set_color("white"); parts["cmedians"].set_linewidth(2)
axes[1].set_xticks(range(len(order)))
axes[1].set_xticklabels(order, rotation=10)
axes[1].set_title("Fare Violin Plot by Period")
axes[1].set_ylabel("Fare ($)")
 
fig.tight_layout()
fig.savefig(f"{OUT}/09_fare_by_period.png", bbox_inches="tight")
plt.close(fig)
 

Plot 9: Rush vs Off-Peak...


In [15]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 10 — Correlation Heatmap
# ══════════════════════════════════════════════════════════════════════
print("Plot 10: Correlation Heatmap...")
numeric_cols = ["fare_amount","distance_km","passenger_count","hour","dow",
                "month","year","is_rush","is_night","is_weekend"]
corr = df[numeric_cols].corr()
 
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, ax=ax, linewidths=0.5, linecolor="#E8E8E8",
            annot_kws={"size": 9},
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold", pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)
 
fig.tight_layout()
fig.savefig(f"{OUT}/10_correlation_heatmap.png", bbox_inches="tight")
plt.close(fig)
 

Plot 10: Correlation Heatmap...


In [16]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 11 — Distance Distribution & Fare per km
# ══════════════════════════════════════════════════════════════════════
print("Plot 11: Distance Analysis...")
df["fare_per_km"] = (df["fare_amount"] / df["distance_km"].clip(0.1)).clip(0, 30)
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle("Trip Distance Analysis", fontsize=14, fontweight="bold")
 
axes[0].hist(df["distance_km"].clip(0, 50), bins=80, color=TEAL,
             edgecolor="white", linewidth=0.3, alpha=0.9)
axes[0].axvline(df["distance_km"].mean(),   color=RED,    lw=2, ls="--",
                label=f"Mean  {df['distance_km'].mean():.1f} km")
axes[0].axvline(df["distance_km"].median(), color=ORANGE, lw=2, ls=":",
                label=f"Median {df['distance_km'].median():.1f} km")
axes[0].set_title("Distance Distribution")
axes[0].set_xlabel("Distance (km)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8.5)
 
# Distance bins vs fare
bins     = [0, 2, 5, 10, 20, 35, 100]
labels_b = ["<2km","2–5","5–10","10–20","20–35",">35km"]
df["dist_bin"] = pd.cut(df["distance_km"], bins=bins, labels=labels_b)
bin_fare = df.groupby("dist_bin", observed=True)["fare_amount"].mean()
axes[1].bar(bin_fare.index, bin_fare.values, color=PURPLE,
            edgecolor="white", linewidth=0.5, alpha=0.9)
axes[1].set_title("Mean Fare by Distance Band")
axes[1].set_xlabel("Distance Band")
axes[1].set_ylabel("Mean Fare ($)")
for i, v in enumerate(bin_fare.values):
    axes[1].text(i, v + 0.3, f"${v:.0f}", ha="center", fontsize=9, fontweight="bold")
 
# Fare per km
axes[2].hist(df["fare_per_km"].clip(0, 15), bins=80, color=PINK,
             edgecolor="white", linewidth=0.3, alpha=0.9)
axes[2].axvline(df["fare_per_km"].mean(), color=RED, lw=2, ls="--",
                label=f"Mean ${df['fare_per_km'].mean():.2f}/km")
axes[2].set_title("Fare per km Distribution")
axes[2].set_xlabel("$/km")
axes[2].set_ylabel("Count")
axes[2].legend(fontsize=8.5)
 
fig.tight_layout()
fig.savefig(f"{OUT}/11_distance_analysis.png", bbox_inches="tight")
plt.close(fig)
 

Plot 11: Distance Analysis...


In [17]:
# ══════════════════════════════════════════════════════════════════════
# PLOT 12 — EDA Summary Dashboard
# ══════════════════════════════════════════════════════════════════════
print("Plot 12: Summary Dashboard...")
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor("#F1F5F9")
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.42, wspace=0.38,
                       top=0.88, bottom=0.08, left=0.07, right=0.97)
 
fig.suptitle("🚕  Uber NYC Fare Prediction — EDA Summary Dashboard",
             fontsize=16, fontweight="bold", y=0.96)
 
# KPI cards (top row)
kpis = [
    (f"{len(df):,}", "Total Rides"),
    (f"${df['fare_amount'].mean():.2f}", "Mean Fare"),
    (f"{df['distance_km'].mean():.1f} km", "Mean Distance"),
    (f"${df['fare_amount'].std():.2f}", "Fare Std Dev"),
]
kpi_colors = [BLUE, GREEN, ORANGE, PURPLE]
for i, ((val, lbl), col) in enumerate(zip(kpis, kpi_colors)):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(col)
    ax.text(0.5, 0.60, val, ha="center", va="center",
            fontsize=17, fontweight="bold", color="white",
            transform=ax.transAxes)
    ax.text(0.5, 0.22, lbl, ha="center", va="center",
            fontsize=9, color="white", alpha=0.88,
            transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
 
# Fare by hour (row 2 left)
ax1 = fig.add_subplot(gs[1, :2])
ax1.bar(hourly["hour"], hourly["mean"], color=BLUE, alpha=0.75, width=0.7)
ax1.plot(hourly["hour"], hourly["median"], "o-", color=ORANGE, lw=2, ms=4)
ax1.axvspan(7,  9,  alpha=0.12, color=RED)
ax1.axvspan(17, 19, alpha=0.12, color=RED)
ax1.set_title("Mean Fare by Hour")
ax1.set_xlabel("Hour"); ax1.set_ylabel("$")
ax1.set_xticks(range(0, 24, 3))
ax1.set_xticklabels([f"{h:02d}:00" for h in range(0,24,3)], fontsize=7.5)
 
# Fare by DOW (row 2 right)
ax2 = fig.add_subplot(gs[1, 2:])
colors_d = [RED if d >= 5 else BLUE for d in dow_stats["dow"]]
ax2.bar([DOW_LABELS[d] for d in dow_stats["dow"]], dow_stats["mean"],
        color=colors_d, alpha=0.85, edgecolor="white")
ax2.set_title("Mean Fare by Day")
ax2.set_ylabel("$")
 
# Distance vs fare (row 3 left)
s2 = df.sample(min(3000, len(df)), random_state=5)
ax3 = fig.add_subplot(gs[2, :2])
ax3.scatter(s2["distance_km"], s2["fare_amount"],
            c=s2["hour"], cmap="plasma", alpha=0.3, s=5)
z2 = np.polyfit(s2["distance_km"], s2["fare_amount"], 1)
xl = np.linspace(0, s2["distance_km"].quantile(0.98), 80)
ax3.plot(xl, np.polyval(z2, xl), color=RED, lw=2)
ax3.set_xlim(0, s2["distance_km"].quantile(0.98))
ax3.set_ylim(0, s2["fare_amount"].quantile(0.98))
r_val = float(df["distance_km"].corr(df["fare_amount"]))
ax3.set_title(f"Distance vs Fare  (r={r_val:.2f})")
ax3.set_xlabel("Distance (km)"); ax3.set_ylabel("Fare ($)")
 
# Passenger pie (row 3 right)
ax4 = fig.add_subplot(gs[2, 2])
ax4.pie(pax_counts.values,
        labels=[str(p) for p in pax_counts.index],
        colors=PALETTE[:len(pax_counts)],
        wedgeprops=dict(edgecolor="white", linewidth=1.2),
        autopct="%1.0f%%", pctdistance=0.78, textprops={"fontsize": 8})
ax4.set_title("Pax Share")
 
# Monthly trend (row 3 far right)
ax5 = fig.add_subplot(gs[2, 3])
ax5.plot(monthly["month"], monthly["mean"], "o-", color=GREEN, lw=2, ms=5)
ax5.fill_between(monthly["month"], monthly["mean"], alpha=0.15, color=GREEN)
ax5.set_xticks(range(1, 13))
ax5.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"], fontsize=7.5)
ax5.set_title("Fare by Month")
ax5.set_ylabel("$")
 
fig.savefig(f"{OUT}/12_eda_dashboard.png", bbox_inches="tight", dpi=140)
plt.close(fig)

Plot 12: Summary Dashboard...
